# SongGraveyard — ACE-Step 1.5 免费云端 Endpoint (Colab)

在 Colab 免费 GPU(T4)上跑起 ACE-Step 1.5 的 REST API,并用 cloudflared 隧道暴露成一个**公网 URL**,队友 / 后端可以直接调用。

主力用 **Cover 模式**:你们的 ghost / resurrect 就是同一个 Cover 任务、不同的 `audio_cover_strength`:
- **Ghost** → strength ≈ 0.4(空灵、自由发挥)
- **Resurrect** → strength ≈ 0.85(忠实跟随原动机)

**免费 Colab 的代价(务必知道):**
- Colab 会因空闲或超时(约 12h)断开,断了 endpoint 就没了 —— 适合开发联调,**不适合 demo 当天的稳定服务**。
- 每次重启,cloudflared 的公网 URL 都会变,要把新 URL 发给队友。
- 首次启动会自动下载几个 G 的模型,耐心等。
- T4 16GB:用默认 turbo 模型 + Cover(跳过大 LM)最稳;Complete 任务需要 base 模型、显存更吃紧(见末尾)。

> 先 Runtime → Change runtime type → 选 **GPU (T4)**,然后从上到下逐格运行。

In [ ]:
# 1. 确认拿到 GPU
!nvidia-smi

In [ ]:
# 2. 安装 ACE-Step 1.5(首次约几分钟)
!pip -q install uv
!git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!uv sync
%cd /content
print("✅ 安装完成")

In [ ]:
# 3. 下载 cloudflared(免费隧道,无需账号)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("✅ cloudflared 就绪")

In [ ]:
# 4. 后台启动 ACE-Step REST API(修正版:覆盖 Colab 的 matplotlib backend)
import subprocess, os, time, requests

# 如果之前那次起失败的进程还在,先清掉,避免端口占用
try:
    server.terminate()
except Exception:
    pass

env = os.environ.copy()
env.update({
    "MPLBACKEND": "Agg",
    "ACESTEP_USE_FLASH_ATTENTION": "false",
    "ACESTEP_INIT_LLM": "false",            # ★ 不加载大 LM(Cover 不需要),省掉 3.7GB RAM
    "ACESTEP_API_HOST": "0.0.0.0",
    "ACESTEP_API_PORT": "8001",
    "ACESTEP_CONFIG_PATH": "acestep-v15-turbo",
    "ACESTEP_OFFLOAD_TO_CPU": "false",      # ★ Colab RAM 太小,offload 反而占 RAM,关掉
    "ACESTEP_LM_BACKEND": "pt",
})

logf = open("/content/acestep_api.log", "w")
server = subprocess.Popen(["uv", "run", "acestep-api"],
                          cwd="/content/ACE-Step-1.5", env=env,
                          stdout=logf, stderr=subprocess.STDOUT)

base_local = "http://127.0.0.1:8001"
ok = False
for i in range(240):                                # 最多等 ~20 分钟(首次下模型)
    try:
        if requests.get(base_local + "/health", timeout=3).status_code == 200:
            ok = True; print("✅ API server 已就绪"); break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print("⚠️ 还没就绪,日志末尾:")
    print(open("/content/acestep_api.log").read()[-3000:])

In [ ]:
# 5. 起 cloudflared 隧道,拿到公网 URL
import subprocess, re, threading, time

PUBLIC_URL = None
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def _watch():
    global PUBLIC_URL
    for line in tunnel.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m and not PUBLIC_URL:
            PUBLIC_URL = m.group(0)

threading.Thread(target=_watch, daemon=True).start()
for _ in range(30):
    if PUBLIC_URL: break
    time.sleep(1)

print("🌍 公网 endpoint(发给队友 / 填进后端):", PUBLIC_URL)
print("   健康检查:", (PUBLIC_URL + "/health") if PUBLIC_URL else "(没拿到就重跑本格)")

## 调用方式

ACE-Step 是**异步** API:
1. `POST /release_task` 提交任务 → 拿 `task_id`
2. `POST /query_result` 轮询直到 `status == 1`
3. `GET /v1/audio?path=...` 下载音频

下面的 `generate_cover()` 把这三步包好了。Ghost / Resurrect 只是 strength 与 prompt 不同。

In [ ]:
# 6. 通用客户端 + 测试(防 OOM + submit 自动重试版)
import requests, json, time
from google.colab import files
from IPython.display import Audio, display

BASE = PUBLIC_URL

def generate_cover(src_audio_path, prompt, strength, lyrics="",
                   duration=15, steps=8, vocal_language="en", base=None, max_wait=1800):
    base = base or BASE
    data = {
        "task_type": "cover", "prompt": prompt, "lyrics": lyrics,
        "vocal_language": vocal_language,
        "audio_cover_strength": str(strength),
        "audio_duration": str(duration), "inference_steps": str(steps),
        "batch_size": "1",            # ★ T4 防 OOM(默认是 2)
        "audio_format": "mp3",
    }
    # --- 提交:容忍 502 / 超时,等后端恢复 ---
    task_id = None
    for attempt in range(6):
        try:
            with open(src_audio_path, "rb") as f:
                r = requests.post(base + "/release_task", data=data,
                                  files={"src_audio": f}, timeout=180)
            if r.status_code >= 500:
                print(f"  submit 收到 {r.status_code},等 server 恢复…({attempt+1}/6)")
                time.sleep(10); continue
            r.raise_for_status()
            task_id = r.json()["data"]["task_id"]; break
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
            print(f"  submit {type(e).__name__},重试…({attempt+1}/6)"); time.sleep(10)
    if not task_id:
        raise RuntimeError("submit 一直失败 → 先重跑第4格重启 server,并看 /content/acestep_api.log")
    print("task_id:", task_id, "— 生成中,首次较慢请耐心等…")

    # --- 轮询:超时不崩,继续等 ---
    t0 = time.time()
    while time.time() - t0 < max_wait:
        try:
            item = requests.post(base + "/query_result",
                                 json={"task_id_list": [task_id]}, timeout=120).json()["data"][0]
        except Exception as e:
            print(f"  …仍在生成({type(e).__name__}),已等 {int(time.time()-t0)}s"); time.sleep(5); continue
        st = item["status"]
        if st == 1:
            res = json.loads(item["result"])[0]
            audio = requests.get(base + res["file"], timeout=180).content
            print(f"  ✅ 完成,用时 {int(time.time()-t0)}s"); return audio, res
        if st == 2:
            raise RuntimeError("生成失败: " + str(item))
        print(f"  …status={st},已等 {int(time.time()-t0)}s"); time.sleep(5)
    raise TimeoutError("超过 max_wait,去看 /content/acestep_api.log")

up = files.upload()
motif = list(up.keys())[0]

ghost_audio, _ = generate_cover(
    motif,
    prompt="ghostly ambient reinterpretation, fragile, distant, reverb-heavy, dreamy, preserving the original mood",
    strength=0.4)
open("ghost.mp3","wb").write(ghost_audio); print("✅ ghost.mp3")

resurrect_audio, _ = generate_cover(
    motif,
    prompt="warm nostalgic city pop, electric piano, smooth bassline, soft drums, night city atmosphere",
    strength=0.85, lyrics="[verse]\nlate night humming on my way home\n")
open("resurrect.mp3","wb").write(resurrect_audio); print("✅ resurrect.mp3")

print("原始动机:"); display(Audio(motif))
print("Ghost:");     display(Audio("ghost.mp3"))
print("Resurrect:"); display(Audio("resurrect.mp3"))

## 队友 / 后端怎么调(把这段塞进你们的 FastAPI 后端)

```python
import requests, json, time

BASE = "https://xxxx.trycloudflare.com"   # 换成上面打印的公网 URL

def acestep_cover(src_path, prompt, strength, lyrics="", duration=30):
    with open(src_path, "rb") as f:
        r = requests.post(f"{BASE}/release_task",
            data={"task_type": "cover", "prompt": prompt, "lyrics": lyrics,
                  "audio_cover_strength": str(strength),
                  "audio_duration": str(duration), "inference_steps": "8",
                  "audio_format": "mp3"},
            files={"src_audio": f}, timeout=180)
    tid = r.json()["data"]["task_id"]
    while True:
        item = requests.post(f"{BASE}/query_result",
                 json={"task_id_list": [tid]}, timeout=60).json()["data"][0]
        if item["status"] == 1:
            res = json.loads(item["result"])[0]
            return requests.get(BASE + res["file"], timeout=180).content   # 返回 mp3 bytes
        if item["status"] == 2:
            raise RuntimeError("failed")
        time.sleep(3)
```

- Ghost = `acestep_cover(path, "<空灵 prompt>", 0.4)`
- Resurrect = `acestep_cover(path, "<风格 prompt>", 0.85, lyrics="...")`
- 多动机 Remix:先用 ffmpeg/pydub 把两段动机拼/混成一条音频,再当 `src_path` 传进来。

### 想做"发展成更完整的 piece"(Complete 任务)
Complete 只在 **base 模型**上有,T4 显存可能不够。要试就把第 4 格的 `ACESTEP_CONFIG_PATH` 改成 `acestep-v15-base`(或调 `POST /v1/init` 加载 base),请求里改 `task_type="complete"`。建议这个放 SuperPOD 或更大的卡上跑。

### 保活 & 兜底
- 保持 Colab 标签页打开、别让电脑休眠。
- 断线后:重跑「第 4、5 格」,把**新的公网 URL** 发给队友(URL 每次会变)。
- demo 当天别赌 Colab —— 用 SuperPOD 预生成好的结果做 fallback。